# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
TASK_ID='task061'
VERIFY_SCOPE='visible'
MODEL_VERSION='task061-symbolic-v1'

In [2]:
# ONNX dependency setup.
import importlib.util, subprocess, sys
required = {'onnx':'onnx', 'onnxruntime':'onnxruntime', 'sklearn':'scikit-learn', 'torch':'torch'}
missing = [pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import onnx, onnxruntime as ort
print('onnx:', onnx.__version__)
print('onnxruntime:', ort.__version__)

Installing missing packages: ['onnxruntime']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 45.4 MB/s eta 0:00:00
onnx: 1.20.1
onnxruntime: 1.27.0


In [3]:

import json, os, zipfile, subprocess, sys, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
CH=10
FORBIDDEN={'Loop','Scan','NonZero','Unique','Script','Function'}

def find_task_json(task_id):
    candidates=[Path.cwd()/f'{task_id}.json',Path('/mnt/data')/f'{task_id}.json',Path('/kaggle/working')/f'{task_id}.json']
    for p in candidates:
        if p.exists(): return p
    for base in [Path('/kaggle/input'),Path.cwd(),Path('/kaggle/working')]:
        if base.exists():
            hits=list(base.rglob(f'{task_id}.json'))
            if hits: return hits[0]
    raise FileNotFoundError(task_id)

def load_task(task_id):
    p=find_task_json(task_id)
    with open(p) as f: return json.load(f),p

def examples_for_scope(task,scope='visible'):
    if scope=='all': return task.get('train',[])+task.get('test',[])+task.get('arc-gen',[])
    return task.get('train',[])+task.get('test',[])

def grid_to_tensor(grid,H,W):
    arr=np.zeros((1,CH,H,W),np.float32)
    for r,row in enumerate(grid[:H]):
        for c,v in enumerate(row[:W]): arr[0,int(v),r,c]=1.0
    return arr

def validate_model(model_path,task,H,W,scope='visible'):
    sess=ort.InferenceSession(str(model_path),providers=['CPUExecutionProvider'])
    right=0; total=0; first_wrong=None; first_wrong_pixels=None
    for i,ex in enumerate(examples_for_scope(task,scope)):
        pred=(sess.run(['output'],{'input':grid_to_tensor(ex['input'],H,W)})[0]>0.5).astype(np.float32)
        exp=grid_to_tensor(ex['output'],H,W)
        total+=1
        if np.array_equal(pred,exp): right+=1
        elif first_wrong is None:
            first_wrong=i; first_wrong_pixels=int(np.sum(pred!=exp))
    m=onnx.load(str(model_path)); ops={}
    for n in m.graph.node: ops[n.op_type]=ops.get(n.op_type,0)+1
    return {'right':right,'total':total,'first_wrong':first_wrong,'first_wrong_pixels':first_wrong_pixels,'file_size_bytes':Path(model_path).stat().st_size,'under_1_4mb':Path(model_path).stat().st_size<1_400_000,'forbidden_ops_present':sorted(FORBIDDEN & set(ops)),'op_counts':ops}


def _onnx_dims(value_info):
    return [d.dim_value for d in value_info.type.tensor_type.shape.dim]

def wrap_exported_onnx_to_30x30(src_path, dst_path):
    """Wrap a smaller static task-canvas model with 30x30 competition I/O.
    The internal symbolic model sees only its learned task canvas; the public ONNX
    accepts/returns [1,10,30,30], which is what the Kaggle runner expects.
    """
    from onnx import helper, TensorProto, numpy_helper
    model = onnx.load(str(src_path))
    in_shape = _onnx_dims(model.graph.input[0])
    out_shape = _onnx_dims(model.graph.output[0])
    assert in_shape[:2] == [1, CH] and out_shape[:2] == [1, CH], (in_shape, out_shape)
    h, w = int(in_shape[2]), int(in_shape[3])
    assert out_shape[2:] == [h, w], (in_shape, out_shape)
    old_input = model.graph.input[0].name
    old_output = model.graph.output[0].name
    inner_input = 'input_inner'
    inner_output = 'output_inner'
    for node in model.graph.node:
        for i, name in enumerate(node.input):
            if name == old_input:
                node.input[i] = inner_input
        for i, name in enumerate(node.output):
            if name == old_output:
                node.output[i] = inner_output
    del model.graph.input[:]
    model.graph.input.extend([helper.make_tensor_value_info('input', TensorProto.FLOAT, [1, CH, 30, 30])])
    del model.graph.output[:]
    model.graph.output.extend([helper.make_tensor_value_info('output', TensorProto.FLOAT, [1, CH, 30, 30])])
    initializers = list(model.graph.initializer)
    initializers.extend([
        numpy_helper.from_array(np.array([0, 0, 0, 0], dtype=np.int64), 'slice_starts'),
        numpy_helper.from_array(np.array([1, CH, h, w], dtype=np.int64), 'slice_ends'),
        numpy_helper.from_array(np.array([0, 1, 2, 3], dtype=np.int64), 'slice_axes'),
        numpy_helper.from_array(np.array([1, 1, 1, 1], dtype=np.int64), 'slice_steps'),
    ])
    nodes = [helper.make_node('Slice', ['input', 'slice_starts', 'slice_ends', 'slice_axes', 'slice_steps'], [inner_input], name='crop_input_to_task_canvas')]
    nodes.extend(list(model.graph.node))
    current = inner_output
    if w < 30:
        initializers.append(numpy_helper.from_array(np.zeros((1, CH, h, 30 - w), dtype=np.float32), 'right_zero_pad'))
        nodes.append(helper.make_node('Concat', [current, 'right_zero_pad'], ['output_width30'], axis=3, name='pad_right_to_30'))
        current = 'output_width30'
    if h < 30:
        initializers.append(numpy_helper.from_array(np.zeros((1, CH, 30 - h, 30), dtype=np.float32), 'bottom_zero_pad'))
        nodes.append(helper.make_node('Concat', [current, 'bottom_zero_pad'], ['output'], axis=2, name='pad_bottom_to_30'))
        current = 'output'
    if current != 'output':
        nodes.append(helper.make_node('Identity', [current], ['output'], name='identity_output'))
    del model.graph.node[:]
    model.graph.node.extend(nodes)
    del model.graph.initializer[:]
    model.graph.initializer.extend(initializers)
    model.ir_version = 8
    onnx.checker.check_model(model)
    onnx.save(model, str(dst_path))
    return {'internal_shape': [1, CH, h, w], 'public_shape': [1, CH, 30, 30]}


In [4]:

class GridResidueRepair(nn.Module):
    def __init__(self,H,W,candidates=tuple(range(2,10))):
        super().__init__(); self.H=H; self.W=W; self.candidates=candidates
        for p in candidates:
            Sr=torch.zeros(H,H); Sc=torch.zeros(W,W)
            for r in range(H):
                for h in range(H):
                    if h%p==r%p: Sr[r,h]=1.0
            for c in range(W):
                for w in range(W):
                    if w%p==c%p: Sc[c,w]=1.0
            self.register_buffer(f'Sr_{p}',Sr); self.register_buffer(f'Sc_{p}',Sc)
    def select_rows(self,t,S): return torch.matmul(t.permute(0,1,3,2),S.t()).permute(0,1,3,2)
    def select_cols(self,t,S): return torch.matmul(t,S.t())
    def cand(self,colors,p):
        Sr=getattr(self,f'Sr_{p}'); Sc=getattr(self,f'Sc_{p}')
        seen=(self.select_cols(self.select_rows(colors,Sr),Sc)>0.5).float()
        dist=torch.amax(seen.sum(dim=1,keepdim=True),dim=(2,3),keepdim=True)
        valid=(dist<=1.5).float()
        bg=(colors.sum(dim=1,keepdim=True)<0.5).float(); any_seen=(seen.sum(dim=1,keepdim=True)>0.5).float()
        out=colors*(1-bg*any_seen)+seen*(bg*any_seen)
        bg2=(out.sum(dim=1,keepdim=True)<0.5).float()
        return torch.cat([bg2,out],dim=1),valid
    def forward(self,x):
        colors=x[:,1:]
        acc=torch.zeros_like(x); remain=torch.ones((x.shape[0],1,1,1),device=x.device,dtype=x.dtype)
        for p in self.candidates:
            pred,valid=self.cand(colors,p)
            pick=remain*valid; acc=acc+pred*pick; remain=remain*(1-pick)
        return acc+x*remain


In [5]:
task,task_path=load_task(TASK_ID)
H=18; W=18
OUT_DIR=Path.cwd()/f'working_submission_{TASK_ID}'
OUT_DIR.mkdir(parents=True,exist_ok=True)
MODEL_PATH=OUT_DIR/f'{TASK_ID}.onnx'
print('task path:',task_path)
print('train:',len(task.get('train',[])),'test:',len(task.get('test',[])),'arc-gen:',len(task.get('arc-gen',[])))
print('internal symbolic shape:',H,W)
print('public ONNX I/O shape:',30,30)
print('MODEL_PATH:',MODEL_PATH)

task path: /kaggle/input/competitions/neurogolf-2026/task061.json
train: 4 test: 1 arc-gen: 262
internal symbolic shape: 18 18
public ONNX I/O shape: 30 30
MODEL_PATH: /kaggle/working/working_submission_task061/task061.onnx


In [6]:
# Build ONNX model and enforce competition constraints.
# Export the symbolic model on its compact internal canvas, then wrap it
# with [1,10,30,30] public I/O for Kaggle submission compatibility.
model=GridResidueRepair(H,W)
model.eval()
INTERNAL_MODEL_PATH=OUT_DIR/f'{TASK_ID}_internal_canvas.onnx'
torch.onnx.export(model, torch.zeros(1,CH,H,W,dtype=torch.float32), str(INTERNAL_MODEL_PATH), input_names=['input'], output_names=['output'], opset_version=17, dynamic_axes=None, do_constant_folding=True, dynamo=False)
onnx_model=onnx.load(str(INTERNAL_MODEL_PATH)); onnx_model.ir_version=8; onnx.checker.check_model(onnx_model); onnx.save(onnx_model,str(INTERNAL_MODEL_PATH))
wrap_info=wrap_exported_onnx_to_30x30(INTERNAL_MODEL_PATH, MODEL_PATH)
validation_report=validate_model(MODEL_PATH,task,30,30,VERIFY_SCOPE)
assert validation_report['right']==validation_report['total'], validation_report
assert validation_report['under_1_4mb'], validation_report['file_size_bytes']
assert not validation_report['forbidden_ops_present'], validation_report['forbidden_ops_present']
{'wrap_info':wrap_info,'validation_report':validation_report}


/tmp/ipykernel_16/2454765117.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, torch.zeros(1,CH,H,W,dtype=torch.float32), str(INTERNAL_MODEL_PATH), input_names=['input'], output_names=['output'], opset_version=17, dynamic_axes=None, do_constant_folding=True, dynamo=False)


{'wrap_info': {'internal_shape': [1, 10, 18, 18],
  'public_shape': [1, 10, 30, 30]},
 'validation_report': {'right': 5,
  'total': 5,
  'first_wrong': None,
  'first_wrong_pixels': None,
  'file_size_bytes': 68359,
  'under_1_4mb': True,
  'forbidden_ops_present': [],
  'op_counts': {'Slice': 2,
   'Identity': 8,
   'Constant': 57,
   'Transpose': 9,
   'MatMul': 16,
   'Greater': 16,
   'Cast': 33,
   'ReduceSum': 17,
   'ReduceMax': 8,
   'LessOrEqual': 8,
   'Less': 9,
   'Mul': 49,
   'Sub': 16,
   'Add': 17,
   'Concat': 10}}}

In [7]:
visible_report=validate_model(MODEL_PATH,task,30,30,'visible')
print('visible_report:',visible_report)

visible_report: {'right': 5, 'total': 5, 'first_wrong': None, 'first_wrong_pixels': None, 'file_size_bytes': 68359, 'under_1_4mb': True, 'forbidden_ops_present': [], 'op_counts': {'Slice': 2, 'Identity': 8, 'Constant': 57, 'Transpose': 9, 'MatMul': 16, 'Greater': 16, 'Cast': 33, 'ReduceSum': 17, 'ReduceMax': 8, 'LessOrEqual': 8, 'Less': 9, 'Mul': 49, 'Sub': 16, 'Add': 17, 'Concat': 10}}


In [8]:
manifest={'task_id':TASK_ID,'model_version':MODEL_VERSION,'rule':'adaptive 2D residue-period repair','validation_report':validation_report,'visible_report':visible_report}
manifest_path=OUT_DIR/f'{TASK_ID}_manifest.json'
with open(manifest_path,'w') as f: json.dump(manifest,f,indent=2)
zip_path=OUT_DIR/f'{TASK_ID}_submission.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf: zf.write(MODEL_PATH,MODEL_PATH.name)
print('wrote',MODEL_PATH)
print('wrote',manifest_path)
print('wrote',zip_path)

submission_path=Path.cwd()/'submission.zip'
with zipfile.ZipFile(submission_path,'w',zipfile.ZIP_DEFLATED) as zf: zf.write(MODEL_PATH, MODEL_PATH.name)

wrote /kaggle/working/working_submission_task061/task061.onnx
wrote /kaggle/working/working_submission_task061/task061_manifest.json
wrote /kaggle/working/working_submission_task061/task061_submission.zip
